# MediChain in Google Colab
Builds and runs both Spring Boot services inside the Colab machine, then opens the dashboard. Run the cells in order. The first build downloads the Spring libraries and takes a few minutes.

Sign in with **admin / Admin@123** (district Vellore).

In [ ]:
# 1. Upload medichain.zip when asked
from google.colab import files
import os, zipfile
if not os.path.isdir('/content/medichain'):
    up = files.upload()
    zipfile.ZipFile(next(iter(up))).extractall('/content')
print(sorted(os.listdir('/content/medichain')))

In [ ]:
# 2. Install Java 17 and Maven 3.9
import os, subprocess
subprocess.run('apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null', shell=True, check=True)
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
mvn_dir = '/opt/apache-maven-3.9.9'
if not os.path.isdir(mvn_dir):
    for url in ['https://archive.apache.org/dist/maven/maven-3/3.9.9/binaries/apache-maven-3.9.9-bin.tar.gz',
                'https://dlcdn.apache.org/maven/maven-3/3.9.9/binaries/apache-maven-3.9.9-bin.tar.gz']:
        if subprocess.run(f'wget -q {url} -O /tmp/maven.tgz && tar -xzf /tmp/maven.tgz -C /opt', shell=True).returncode == 0:
            break
    else:
        print('Maven download failed, using the apt package instead')
        subprocess.run('apt-get -qq install -y maven > /dev/null', shell=True, check=True)
        mvn_dir = '/usr/share/maven'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + mvn_dir + '/bin:' + os.environ['PATH']
!java -version
!mvn -version | head -2

In [ ]:
# 3. Build both services (prints BUILD OK or the last lines of the error)
import subprocess
built = True
for name in ['inventory-service', 'forecast-service']:
    r = subprocess.run('mvn -B -q -DskipTests package', shell=True, cwd=f'/content/medichain/{name}', capture_output=True, text=True)
    print(name, 'BUILD OK' if r.returncode == 0 else 'BUILD FAILED')
    if r.returncode != 0:
        built = False
        print((r.stdout + r.stderr)[-4000:])
!ls -la /content/medichain/*/target/*.jar

In [ ]:
# 4. Start both services and wait until they answer
import subprocess, time, urllib.request
subprocess.run("pkill -f 'service-1.0.0.jar' ; pkill -f medichain-lite.jar ; true", shell=True)
time.sleep(2)
for name in ['inventory', 'forecast']:
    subprocess.Popen(f'nohup java -jar /content/medichain/{name}-service/target/{name}-service-1.0.0.jar > /content/{name}.log 2>&1 &', shell=True)
    time.sleep(25 if name == 'inventory' else 5)
ok = False
for _ in range(60):
    try:
        print(urllib.request.urlopen('http://localhost:8082/api/health').read().decode()); ok = True; break
    except Exception:
        time.sleep(3)
if not ok:
    print('forecast-service did not start. Last log lines:')
    print(open('/content/forecast.log').read()[-3000:])
    print('inventory-service log:')
    print(open('/content/inventory.log').read()[-3000:])

In [ ]:
# 5. Open the dashboard in a new tab
from google.colab import output
output.serve_kernel_port_as_window(8082, anchor_text='Open MediChain')

In [ ]:
# Optional: show it inside the notebook instead
from google.colab import output
output.serve_kernel_port_as_iframe(8082, height=900)

## If the Spring Boot build or start-up fails
Copy the error printed by cell 3 or 4 so it can be fixed. To see the dashboard meanwhile, the cell below runs the Java-only build (`lite/medichain-lite.jar`): same screens and analysis code, but without Spring Boot or the H2 database.

In [ ]:
# Fallback only: Java-only build on port 8082
import subprocess, time, urllib.request
from google.colab import output
subprocess.run("pkill -f 'service-1.0.0.jar' ; pkill -f medichain-lite.jar ; true", shell=True)
time.sleep(2)
subprocess.Popen('nohup java -jar /content/medichain/lite/medichain-lite.jar 8082 nobrowser > /content/lite.log 2>&1 &', shell=True)
time.sleep(8)
print(urllib.request.urlopen('http://localhost:8082/api/health').read().decode())
output.serve_kernel_port_as_window(8082, anchor_text='Open MediChain (Java-only build)')